In [1]:
# make sure jupyter server is installed in the environment
# then install dependencies
%pip install pandas nltk scikit-learn numpy matplotlib symspellpy setuptools --quiet

from config import get_merged_dataframe
from main import configure

configure()

df = get_merged_dataframe(
    './data/processedNegative.csv',
    './data/processedPositive.csv',
    './data/processedNeutral.csv',
)

df.sample(10).reset_index(drop=True)

Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt_tab to /home/samy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/samy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/samy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,tweet,sentiment
0,i want harry's phonecase sad,negative
1,Joy we should get him back,negative
2,Modi's bug,neutral
3,the sun is shining happy,positive
4,and more. Also in epaper,neutral
5,The rallying cry of has become a thing to beho...,neutral
6,veggies,neutral
7,not being spe,positive
8,Shameful,negative
9,mom please stop making cookies,negative


In [2]:
from tokenizer import (
    lemmatize_tokens,
    stem_tokens,
    snowball_stem_tokens,
    lancaster_stem_tokens,
    misspell_and_lemmatize_tokens,
    misspell_tokens
)
from vectorizer import tfidf_vectorize, count_vectorize, binary_vectorize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB, ComplementNB

config = {
    "tokenization": {
        "Tokenization": None,
        "Lemmatization": lemmatize_tokens,
        "Stemming": stem_tokens,
        "Stemming Snowball": snowball_stem_tokens,
        "Stemming Lancaster": lancaster_stem_tokens,
        "Misspellings": misspell_tokens,
        "Misspellings + Lemmatization": misspell_and_lemmatize_tokens,
    },
    "vectorization": {
        "TF-IDF": tfidf_vectorize,
        "Count": count_vectorize,
        "Binary Count": binary_vectorize,
    },
    "classifiers": [
        LogisticRegression(),
        RandomForestClassifier(),
        MultinomialNB(),
        SVC(),
        BernoulliNB(),
        ComplementNB(),
    ],
}

/home/samy/tweets/tokenizer.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### Classification

In [3]:
from train import train_model, evaluate_model
import warnings
warnings.filterwarnings('ignore', message='The parameter.*token_pattern.*will not be used')

# Grid Search over all combinations
for model in config['classifiers']:
    # For each vectorization technique
    for vectorizer_name, vectorizer_func in config['vectorization'].items():
        # For each tokenization technique
        for tokenizer_name, tokenizer_func in config['tokenization'].items():
            # Print current combination
            print(f"Training {model.__class__.__name__}, {vectorizer_name}, {tokenizer_name}... ")
            model, _, x_test, y_test = train_model(
                df,
                tweet_column='tweet',
                sentiment_column='sentiment',
                tokenizer=tokenizer_func,
                vectorizer=vectorizer_func,
                classifier=model,
            )
            acc = evaluate_model(model, x_test, y_test)
            # multiply by 100 to get percentage
            print(f"Accuracy: {acc:.2%}")
            print("-" * 50)

Training LogisticRegression, TF-IDF, Tokenization... 
Accuracy: 87.72%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Lemmatization... 
Accuracy: 88.58%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Stemming... 
Accuracy: 88.58%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Stemming Snowball... 
Accuracy: 88.44%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Stemming Lancaster... 
Accuracy: 88.73%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Misspellings... 
Accuracy: 87.43%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Misspellings + Lemmatization... 
Accuracy: 86.99%
--------------------------------------------------
Training LogisticRegression, Count, Tokenization... 
Accuracy: 88.73%
--------------------------------------------------
T